In [0]:
# Silver layer tables
orders_df = spark.read.format("delta").table("ecommerce_catalog.silver.orders_clean")
payments_df = spark.read.format("delta").table("ecommerce_catalog.silver.order_payments_clean")
products_df = spark.read.format("delta").table("ecommerce_catalog.silver.products_clean")
customers_df = spark.read.format("delta").table("ecommerce_catalog.silver.customers_clean")
sellers_df = spark.read.format("delta").table("ecommerce_catalog.silver.sellers_clean")

#Step 2: Create Dimension Tables
#2a. Customer Dimension
dim_customers = customers_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix"
).dropDuplicates(["customer_id"])

dim_customers.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.dim_customers")

#2b. Product Dimension
dim_products = products_df.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).dropDuplicates(["product_id"])

dim_products.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.dim_products")

#2c. Seller Dimension
dim_sellers = sellers_df.select(
    "seller_id",
    "seller_city",
    "seller_state"
).dropDuplicates(["seller_id"])

dim_sellers.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.dim_sellers")

#2d. Order Dimension
dim_orders = orders_df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).dropDuplicates(["order_id"])

dim_orders.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.dim_orders")

#2e. Date Dimension
from pyspark.sql.functions import col, year, month, dayofmonth, weekofyear, dayofweek
dim_date = orders_df.select(
    col("order_purchase_timestamp").alias("date")
).dropDuplicates(["date"]) \
  .withColumn("year", year("date")) \
  .withColumn("month", month("date")) \
  .withColumn("day", dayofmonth("date")) \
  .withColumn("week_of_year", weekofyear("date")) \
  .withColumn("day_of_week", dayofweek("date"))
dim_date.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.dim_date")

##Step 3: Create Fact Table

#Fact table combines orders, payments, products, and sellers:

from pyspark.sql.functions import col, coalesce, lit

# Read order_items Silver table
order_items_df = spark.read.format("delta").table("ecommerce_catalog.silver.order_items_clean")

fact_sales = (orders_df.alias("o")
              .join(order_items_df.alias("oi"), col("o.order_id") == col("oi.order_id"), "inner")
              .join(products_df.alias("prd"), col("oi.product_id") == col("prd.product_id"), "left")
              .join(sellers_df.alias("s"), col("oi.seller_id") == col("s.seller_id"), "left")
              .join(payments_df.alias("p"), col("o.order_id") == col("p.order_id"), "left")
              .select(
                  col("o.order_id"),
                  col("o.customer_id"),
                  col("o.order_purchase_timestamp").alias("order_date"),
                  col("oi.product_id"),
                  col("oi.seller_id"),
                  coalesce(col("p.payment_sequential"), lit(1)).alias("payment_seq"),
                  coalesce(col("p.payment_type"), lit("unknown")).alias("payment_type"),
                  coalesce(col("p.payment_installments"), lit(1)).alias("payment_installments"),
                  coalesce(col("p.payment_value"), lit(0.0)).alias("payment_value"),
                  col("oi.price").alias("product_price"),
                  col("oi.freight_value")
              ))

fact_sales.write.format("delta").mode("overwrite").saveAsTable("ecommerce_catalog.gold.fact_sales")


#Business Insightss

#-----------------
# Read Gold layer
#-----------------
from pyspark.sql.functions import *
fact_sales = spark.table("ecommerce_catalog.gold.fact_sales")
dim_customers = spark.table("ecommerce_catalog.gold.dim_customers")
dim_products = spark.table("ecommerce_catalog.gold.dim_products")
dim_sellers = spark.table("ecommerce_catalog.gold.dim_sellers")
dim_orders = spark.table("ecommerce_catalog.gold.dim_orders")
dim_date = spark.table("ecommerce_catalog.gold.dim_date")

#----------------------------
#Low level Business Insights
#----------------------------
#Total Revenue Trend

revenue_trend = fact_sales \
    .groupBy(year("order_date").alias("year"),
             month("order_date").alias("month")) \
    .agg(
        sum(col("product_price") + col("freight_value")).alias("total_revenue")
    )
revenue_trend.show()

#Total Orders and Sales Volume

from pyspark.sql.functions import countDistinct, count
orders_sales = fact_sales.groupBy("order_date") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("items_sold")
    )
orders_sales.show()

#Top Selling Products

top_products = fact_sales \
    .join(dim_products, "product_id") \
    .groupBy("product_category_name") \
    .agg(
        sum("product_price").alias("total_revenue"),
        count("product_id").alias("items_sold")
    ) \
    .orderBy(col("total_revenue").desc())
top_products.show()

#Sales by Product Category

sales_by_category = fact_sales \
    .join(dim_products, "product_id") \
    .groupBy("product_category_name") \
    .agg(
        sum(col("product_price") + col("freight_value")).alias("total_revenue"),
        count("product_id").alias("items_sold")
    ) \
    .orderBy(col("total_revenue").desc())
sales_by_category.show()

#Customer Distribution by Region

customer_distribution = fact_sales \
    .join(dim_customers, "customer_id") \
    .groupBy("customer_state", "customer_city") \
    .agg(
        countDistinct("customer_id").alias("total_customers"),
        countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(col("total_customers").desc())

customer_distribution.show()

#===========================================

#------------------------------
#Medium levelBusiness Insights
#------------------------------
from pyspark.sql.functions import *

#Customer Purchase Frequency

customer_purchase = fact_sales \
    .groupBy("customer_id") \
    .agg(
        countDistinct("order_id").alias("total_orders")
    )
customer_purchase.show()

#Average Order Value (AOV)

aov = fact_sales \
    .agg(
        (sum("payment_value") / countDistinct("order_id")).alias("average_order_value")
    )
aov.show()

#Seller Performance Analysis

seller_performance = fact_sales \
    .join(dim_sellers, "seller_id") \
    .groupBy("seller_id", "seller_state") \
    .agg(
        sum("product_price").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(col("total_revenue").desc())
seller_performance.show()

#Delivery Performance

#Join Orders with Sales
delivery_df = fact_sales \
    .join(dim_orders, "order_id")

#Average Delivery Time    
avg_delivery = delivery_df \
    .withColumn(
        "delivery_days",
        datediff("order_delivered_customer_date","order_purchase_timestamp")
    ) \
    .agg(
        avg("delivery_days").alias("avg_delivery_days")
    )
avg_delivery.show()

#Delayed Deliveries

delayed_orders = delivery_df \
    .filter(col("order_delivered_customer_date") > col("order_estimated_delivery_date")) \
    .count()

print("Delayed Deliveries:", delayed_orders)

#Shipping Cost Impact on Revenue
shipping_analysis = fact_sales \
    .agg(
        avg("freight_value").alias("avg_freight_value"),
        (sum("freight_value") / sum("product_price") * 100).alias("shipping_cost_percentage")
    )
shipping_analysis.show()

#===============================================

#----------------------------
#Advanced Business Insights
#----------------------------

#Customer Lifetime Value (CLV)
clv = fact_sales \
    .join(dim_customers, "customer_id") \
    .groupBy("customer_id") \
    .agg(
        sum("payment_value").alias("customer_lifetime_value")
    ) \
    .orderBy(col("customer_lifetime_value").desc())
clv.show()

#Customer Segmentation
customer_segmentation = fact_sales \
    .join(dim_customers, "customer_id") \
    .groupBy("customer_id","customer_state") \
    .agg(
        countDistinct("order_id").alias("purchase_frequency"),
        sum("payment_value").alias("total_spent"),
        avg("payment_value").alias("avg_spent")
    )
customer_segmentation.show()


#Product Affinity Analysis (Market Basket)

fs1 = fact_sales.alias("fs1")
fs2 = fact_sales.alias("fs2")
product_affinity = fs1.join(
    fs2,
    (col("fs1.order_id") == col("fs2.order_id")) &
    (col("fs1.product_id") != col("fs2.product_id"))
) \
.groupBy(
    col("fs1.product_id").alias("product_A"),
    col("fs2.product_id").alias("product_B")
) \
.agg(
    count("*").alias("times_bought_together")
) \
.orderBy(col("times_bought_together").desc())
product_affinity.show()

#Demand Forecasting (Historical Trend)

demand_trend = fact_sales \
    .join(dim_products, "product_id") \
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month"),
        "product_category_name"
    ) \
    .agg(
        count("product_id").alias("total_demand")
    ) \
    .orderBy("year","month")
demand_trend.show()

#Customer Churn Identification

customer_last_purchase = fact_sales \
    .groupBy("customer_id") \
    .agg(
        max("order_date").alias("last_purchase_date")
    )
churn_customers = customer_last_purchase \
    .withColumn(
        "days_since_last_purchase",
        datediff(current_date(), col("last_purchase_date"))
    ) \
    .filter(col("days_since_last_purchase") > 180)
churn_customers.show()



